# 07. Paquete de datos para la demostración

La aplicación de la fase 7 corre en una laptop, pero el dataset completo pesa decenas de gigabytes. Por eso este notebook arma un paquete solo con lo que la aplicación necesita, recortado a los artículos disponibles.

| Archivo | Contenido |
|---|---|
| `articles.parquet` | Metadatos de los artículos disponibles |
| `img_emb.npy` y `emb_ids.parquet` | Embeddings de imagen del notebook 03 para esos artículos |
| `complements.parquet` | Complementos del notebook 04 en los que el ancla y el complemento están disponibles |
| `available.parquet` | Catálogo disponible y popularidad, calculados en el notebook 06 |
| `thresholds.json` | Umbrales calibrados en el notebook 06 |
| `images/` | Miniaturas JPEG de 360 por 540 pixeles de cada artículo |

Los inputs son la competencia de H&M, las salidas de los notebooks 01, 03, 04 y 06, y el dataset `hm-stylist-src`. No necesita GPU. El resultado es `hm-demo-data.zip`, que se descomprime en la laptop y se le indica a la aplicación dónde está.

## Carga del catálogo
El paquete se arma en una carpeta temporal fuera de `/kaggle/working`, porque Kaggle limita el número de archivos de salida y solo queremos guardar el zip final.

In [1]:
import os, sys, json, time, shutil, zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from PIL import Image

def find_dir_with(fname, root):
    for d, dirs, files in os.walk(root):
        dirs[:] = [x for x in dirs if x != "images"]
        if fname in files:
            return Path(d)

SRC = find_dir_with("stylist.py", "/kaggle/input")
assert SRC is not None, "Agrega el dataset hm-stylist-src como Input"
sys.path.insert(0, str(SRC))
from data import load_catalog, find

OUT = Path("/kaggle/working")
PKG = Path("/tmp/hm-demo")            # se arma fuera de /kaggle/working y solo se guarda el zip
shutil.rmtree(PKG, ignore_errors=True)
(PKG / "images").mkdir(parents=True)

t0 = time.time()
cat = load_catalog()                  # usa available.parquet del 06 (mismo catálogo que se evaluó)
assert cat.images_dir is not None, "Agrega la competencia H&M como Input (imágenes)"
avail = cat.avail
print(f"{len(cat.articles):,} artículos · {len(avail):,} disponibles ({time.time() - t0:.0f}s)")

105,542 artículos · 28,966 disponibles (6s)


## Tablas y embeddings recortados
Filtramos los artículos, los embeddings y los complementos para quedarnos solo con los disponibles. Con esto los complementos pasan de 4.4 millones de pares a unos 940 mil.

In [2]:
aid = cat.aid
arts = cat.articles.iloc[avail].reset_index(drop=True)
arts.to_parquet(PKG / "articles.parquet")
np.save(PKG / "img_emb.npy", cat.emb[avail].astype(np.float32))
pd.DataFrame({"article_id": aid[avail]}).to_parquet(PKG / "emb_ids.parquet")

is_av = np.zeros(len(cat.articles), bool); is_av[avail] = True
c = cat.comp[is_av[cat.comp.a.to_numpy()] & is_av[cat.comp.b.to_numpy()]]
pd.DataFrame({"anchor_id": aid[c.a.to_numpy()], "article_id": aid[c.b.to_numpy()],
              "score": c.score.to_numpy(np.float32)}).to_parquet(PKG / "complements.parquet")
cat.availability_table().to_parquet(PKG / "available.parquet")
shutil.copy(find("thresholds.json"), PKG / "thresholds.json")

print(f"Complementos: {len(c):,} de {len(cat.comp):,} pares (los dos artículos disponibles)")
print(json.load(open(PKG / "thresholds.json")))

Complementos: 942,603 de 4,400,154 pares (los dos artículos disponibles)
{'tau_ooc': 0.7629646360874176, 'tau_fashion': 0.6}


## Miniaturas
Reducimos cada foto a 360 por 540 pixeles, que respeta la proporción vertical de las fotos de H&M, y la guardamos en JPEG con calidad 82. Las imágenes se procesan en paralelo y en total ocupan unos 430 MB.

In [3]:
def thumb(a):
    src = cat.image_path(a)
    if src is None:
        return 0
    dst = PKG / "images" / a[:3] / f"{a}.jpg"
    dst.parent.mkdir(exist_ok=True)
    im = Image.open(src)
    im.draft("RGB", (720, 1080))        # decodifica el JPEG a menor resolución: más rápido
    im = im.convert("RGB")
    im.thumbnail((360, 540))
    im.save(dst, "JPEG", quality=82, optimize=True)
    return 1

t0 = time.time()
with ThreadPoolExecutor(os.cpu_count() * 2) as ex:
    done = sum(ex.map(thumb, aid[avail], chunksize=256))
size = sum(f.stat().st_size for f in (PKG / "images").rglob("*.jpg"))
print(f"{done:,} miniaturas · {size / 1e6:.0f} MB · {time.time() - t0:.0f}s")

28,966 miniaturas · 431 MB · 519s


## Verificación
Cargamos el paquete con el mismo código de `src/` que usa la aplicación y pedimos una recomendación sin modelo de lenguaje. Si al paquete le faltara algo, esta celda fallaría.

In [4]:
os.environ["HM_DATA_DIR"] = str(PKG)
os.environ["HM_IMAGES_DIR"] = str(PKG / "images")
import importlib, data
importlib.reload(data)
from ranking import Recommender
from embeddings import FashionCLIP
from stylist import Stylist, Session

demo = data.load_catalog()
rec = Recommender(demo)
th = json.load(open(PKG / "thresholds.json"))
st = Stylist(rec, FashionCLIP(device="cpu"), llm=None, tau_ooc=th["tau_ooc"], tau_fashion=th["tau_fashion"])
assert len(demo.avail) == len(avail) and demo.images_dir == PKG / "images"

a = demo.aid[demo.avail[np.argmax(demo.pop[demo.avail])]]
r = st.chat(Session(), "", demo.load_image(a))
print("Prueba con", a, demo.articles.set_index("article_id").loc[a, "prod_name"], "→", r["flags"])
for x in r["articulos"]:
    print(f"  · [{x['tipo']}] {x['prod_name']} ({x['product_type_name']}, {x['colour_group_name']})")
assert r["articulos"], "La prueba no devolvió artículos"

config.json:   0%|          | 0.00/4.46k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/568 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Prueba con 0909370001 FF PI Haley dress → []
  · [similar] FF PI Haley dress (Dress, Off White)
  · [similar] Mina tunic (Dress, White)
  · [similar] Mindy tunic (Blouse, White)
  · [complemento] Ibsen boot (Boots, Light Beige)
  · [complemento] Joyce high shaft (Boots, Black)
  · [complemento] Flirty Toppy eardeco pk (Earring, Gold)


## Archivo zip
Como las imágenes ya están comprimidas en JPEG, el zip las guarda sin volver a comprimirlas y así se arma mucho más rápido.

In [5]:
t0 = time.time()
zpath = OUT / "hm-demo-data.zip"
with zipfile.ZipFile(zpath, "w") as z:
    for f in sorted(PKG.rglob("*")):
        if f.is_file():
            comp = zipfile.ZIP_STORED if f.suffix == ".jpg" else zipfile.ZIP_DEFLATED
            z.write(f, Path("hm-demo-data") / f.relative_to(PKG), compress_type=comp)
print(f"{zpath.name}: {zpath.stat().st_size / 1e6:.0f} MB ({time.time() - t0:.0f}s)")
for f in sorted(PKG.glob("*")):
    if f.is_file():
        print(f"  {f.name:22s} {f.stat().st_size / 1e6:7.1f} MB")
print(sorted(os.listdir(OUT)))

hm-demo-data.zip: 494 MB (13s)
  articles.parquet           3.7 MB
  available.parquet          0.2 MB
  complements.parquet        2.4 MB
  emb_ids.parquet            0.2 MB
  img_emb.npy               59.3 MB
  thresholds.json            0.0 MB
['hm-demo-data.zip']
